# Adjacency Matrices

In this notebook we will explore how to represent networks using an **adjacency matrix** structure, which can be used for efficient computation of network properties and is used in various network analysis algrotihms.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

### Networks to Matrices

We will use the sample network of LinkedIn connections for this notebook:

In [ ]:
g1 = nx.read_gexf("linkedin25.gexf")

In an **adjacency matrix** representation of a network, the entries in the matrix indicate whether pairs of nodes are connected or not (i.e. whether they share an edge). 

In NetworkX, we can create the matrix representation of a network using the function `adjacency_matrix()`. The output of this function is a SciPy sparse matrix, which is memory-efficient for large networks with relatively few connections:

In [ ]:
A = nx.adjacency_matrix(g1)
print(f"Matrix has {A.shape[0]} rows and {A.shape[1]} columns")

In many cases it might be more useful to produce an adjacency matrix in the form of a NumPy dense matrix using `to_numpy_array()`. This format provides more straightforward access to individual matrix elements and integrates well with NumPy's mathematical operations:

In [ ]:
B = nx.to_numpy_array(g1)
# check the size is the same
print(f"Matrix has {B.shape[0]} rows and {B.shape[1]} columns")
# display the matrix
B

We can use NumPy to store an adjacency matrix for later use (e.g. in CSV format) using the `np.savetxt()` function.

The *delimiter* argument specifies the character separating columns, while the *fmt* argument specifies the precision used to store the matrix values—in this case we do not need any decimal places since we are dealing with binary connectivity data.

In [ ]:
np.savetxt("adjacency-sample.txt", B, delimiter=",", fmt="%0.f")

We can load this matrix back using the `np.loadtxt()` function, recreating the original adjacency matrix structure.

In [ ]:
Z = np.loadtxt("adjacency-sample.txt", delimiter=",")
print(f"Matrix has {Z.shape[0]} rows and {Z.shape[1]} columns")

### Adjacency Matrices and Pandas

We could also create an adjacency matrix as a NumPy array, and then convert it to a Pandas DataFrame, where the row and column indices correspond to the node identifiers. This approach provides convenient labelling and indexing capabilities.

In [ ]:
# create a NumPy adjacency matrix
A1 = nx.to_numpy_array(g1)
print(f"Matrix has {A1.shape[0]} rows and {A1.shape[1]} columns")

In [ ]:
# sort the node identifiers
node_ids = sorted(g1.nodes())
# now create the DataFrame, using the nodelist for indices
df1 = pd.DataFrame(A1, index=node_ids, columns=node_ids)

In [ ]:
df1.head(5)

NetworkX provides a shortcut for performing this process via a single call to the function `nx.to_pandas_adjacency()`, which combines the matrix creation and DataFrame conversion steps:

In [ ]:
df2 = nx.to_pandas_adjacency(g1, nodelist=node_ids)
df2.head(5)

We can run this process in reverse, converting a square Pandas DataFrame (which stores an adjacency matrix) into a network using the function `nx.from_pandas_adjacency()`. 

Note that the indices from the DataFrame are used as node names in the new network, preserving the original node identifiers.

In [ ]:
g1b = nx.from_pandas_adjacency(df2)
print(g1b.nodes())

### Matrices to Networks

We can also convert an adjacency matrix into a network. In the case of a matrix representing an undirected network, this matrix should be symmetric - i.e. the values above the diagonal are a mirror of those below the diagonal, since connections are bidirectional.

In [ ]:
A2 = np.matrix([[0,1,0,0], [1,0,1,1], [0,1,0,1], [0,1,1,0]])
print(f"Matrix has {A2.shape[0]} rows and {A2.shape[1]} columns")
A2

To convert a NumPy matrix, use `from_numpy_array()`. By default, the resulting network will be **undirected**, which is appropriate for symmetric adjacency matrices.

In [ ]:
g2 = nx.from_numpy_array(A2)
type(g2)

By default, the nodes in the new network are integers indexed from 0, corresponding to the rows and columns of the original matrix.

In [ ]:
list(g2.nodes())

In [ ]:
list(g2.edges())

In [ ]:
nx.draw_networkx(g2, 
                 with_labels=True, 
                 node_size=800, 
                 node_color="yellow")
plt.axis("off")
plt.show()

To create a **weighted** network, we simply use non-binary values for the matrix entries. These values will be used to populate the edge *weight* attribute values. Again, for an undirected network the matrix should be symmetric to ensure consistent weight values for bidirectional connections.

In [ ]:
A3 = np.matrix([[0,5,0,0], [5,0,3,4], [0,3,0,1], [0,4,1,0]])
print(f"Matrix has {A3.shape[0]} rows and {A3.shape[1]} columns")
A3

In [ ]:
g3 = nx.from_numpy_array( A3 )

We can check that the edges here are weighted by examining their attributes:

In [ ]:
for e in g3.edges(data=True):
    print(e)

For a **directed network**, the adjacency matrix does not necessarily need to be symmetric—i.e. edges may not be reciprocated, allowing for one-way connections between nodes.

In [ ]:
A4 = np.matrix([[1,1,0,0], [1,0,1,1], [0,1,0,0], [0,1,0,1]])
print(f"Matrix has {A4.shape[0]} rows and {A4.shape[1]} columns")
A4

When calling `from_numpy_array()`, in this case we need to specify that we want to create a directed network using the *create_using* parameter:

In [ ]:
g4 = nx.from_numpy_array(A4, create_using=nx.DiGraph)
type(g4)

In [ ]:
list(g4.edges())

In [ ]:
# draw the network, include arrows to show direction of edges
nx.draw_networkx(g4, 
                 arrows=True, 
                 with_labels=True, 
                 node_size=800, 
                 node_color="yellow")
plt.axis("off")
plt.show()

### Visualising Adjacency Matrices

One approach for visualising a network via an adjacency matrix is to produce a **heatmap**, where the entries in the matrix are represented as colours. This visualisation technique allows us to identify patterns and structures in the network connectivity.

For this purpose we will use the *Seaborn* package which provides additional visualisation capabilities beyond those of *Matplotlib* (https://seaborn.pydata.org/). If the package is not installed on your machine, you can install it in the terminal or command line using:

> pip install seaborn

In [ ]:
import seaborn as sns
# tune the visual settings of Seaborn
sns.set()
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.2)

Firstly, we will visualise an unweighted network via its adjacency matrix - the simple LinkedIn network from above. This will be a binary matrix where two nodes are either connected by an edge (represented by 1) or not connected (represented by 0).

In [ ]:
g5 = nx.read_gexf("linkedin25.gexf")

In this case we will order the node identifiers in the rows and columns alphabetically.

In [ ]:
# get sorted list of nodes IDs
node_ids = sorted(g5.nodes())
print("Ordered nodes:", node_ids)
# create the matrix using the specified ordering
A5 = nx.to_numpy_array(g5, nodelist=node_ids)

In [ ]:
# draw the heatmap using Seaborn
plt.figure(figsize=(14,11))
ax = sns.heatmap(A5, linewidths=0.1, xticklabels=node_ids, yticklabels=node_ids)
plt.show()

### Visualising Weighted Matrices

Next, we will visualise a weighted directed network - in this case, the sample of the US flight network. 

In [ ]:
g6 = nx.read_gexf("airstats-sample.gexf")

For weighted networks, the colour saturation of each cell in the heatmap is proportional to the edge weight, providing intuitive visual representation of connection strength.

Note that for directed networks, the heatmap is asymmetric, reflecting the directional nature of the connections.

In [ ]:
# get sorted list of nodes IDs
node_ids = sorted(g6.nodes())
# create the matrix using the specified ordering
A6 = nx.to_numpy_array(g6, nodelist=node_ids)

In [ ]:
plt.figure(figsize=(14,11))
ax = sns.heatmap(A6, linewidths=0.1, xticklabels=node_ids, yticklabels=node_ids)
plt.show()